In [1]:
%cd ..

/home/bhchen/LearnKalmanGain


In [13]:
import os
import glob
from PIL import Image
from pathlib import Path
import re
from typing import List, Tuple, Dict

def create_gif_robust(image_folder, file_pattern, gif_path, duration=100, max_value=None):
    """
    Function:
        Creates a GIF from images matching a pattern with a numerical wildcard.
        This version is robust and does not depend on any keywords like 'timestep'.
        It identifies the number by what the '*' in the pattern matches.
    Input:
        image_folder (str): The path to the folder containing the images.
        file_pattern (str): Filename pattern with one '*' as a wildcard for a number.
                            Example: "frame_*_render.png"
        gif_path (str): The path to save the output GIF file.
        duration (int): Duration (in milliseconds) for each frame.
        max_value (int, optional): The maximum value of the wildcard part to include.
                                If None, all matched images will be used.
    Output:
        None
    """
    if file_pattern.count('*') != 1:
        print("Error: The file_pattern must contain exactly one '*' wildcard.")
        return

    # Dynamically determine the prefix and suffix from the pattern
    prefix, suffix = file_pattern.split('*')
    
    search_path = os.path.join(image_folder, file_pattern)
    all_filenames = glob.glob(search_path)
    
    if not all_filenames:
        print(f"Error: No images found in '{image_folder}' matching '{file_pattern}'.")
        return

    files_with_values = []
    for f_path in all_filenames:
        basename = os.path.basename(f_path)
        # Extract the part of the filename that corresponds to the wildcard
        if basename.startswith(prefix) and basename.endswith(suffix):
            try:
                # Remove prefix and suffix to get the numerical part
                value_str = basename[len(prefix):-len(suffix)]
                value = int(value_str)
                files_with_values.append({'value': value, 'path': f_path})
            except (ValueError, IndexError):
                # Ignore files where the wildcard part is not a valid integer
                print(f"Warning: Could not extract a valid number from '{basename}'. Skipping.")
                continue

    if not files_with_values:
        print("Error: Found matching files, but could not extract numbers from any of them.")
        return

    # Sort the files based on the extracted numerical value
    files_with_values.sort(key=lambda item: item['value'])

    # Filter by max_value if provided
    if max_value is not None:
        print(f"Filtering frames to include values up to {max_value}.")
        files_with_values = [item for item in files_with_values if item['value'] <= max_value]

    if not files_with_values:
        print(f"Error: After filtering, no images remained with a value <= {max_value}.")
        return
        
    final_filenames = [item['path'] for item in files_with_values]
    print(f"Creating GIF with {len(final_filenames)} frames...")

    images = [Image.open(fn) for fn in final_filenames]
    images[0].save(
        gif_path,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0
    )
    print(f"GIF saved successfully at: {gif_path}")
    
def pair_prior_post(
    image_directory: str,
    pattern: str,
    left_tag: str = "PRIOR",
    right_tag: str = "POST",
    concat_token: str = "PRIOR_POST",
    overwrite: bool = False,
) -> Tuple[int, List[Path]]:
    """
    Find PRIOR/POST image pairs for the same timestep and save a side-by-side concat
    (PRIOR on the left, POST on the right) in the same directory as the sources.
    """

    root = Path(image_directory)
    if not root.exists():
        raise FileNotFoundError(f"Directory not found: {root}")

    # --- Build regex safely ---
    escaped = re.escape(pattern)

    # Ensure we have timestep* to capture digits
    if r"timestep\*" not in escaped:
        raise ValueError("Pattern must contain 'timestep*' so the timestep can be captured.")
    escaped = escaped.replace(r"timestep\*", r"timestep(\d+)")

    # Replace the FIRST occurrence of the escaped right_tag with a capture group (left|right)
    escaped_right = re.escape(right_tag)
    escaped_left = re.escape(left_tag)

    if escaped_right not in escaped:
        raise ValueError(
            f"Could not find the tag '{right_tag}' inside the pattern. "
            f"Please ensure pattern includes it exactly once."
        )

    escaped = escaped.replace(escaped_right, f"({escaped_left}|{escaped_right})", 1)

    name_re = re.compile(rf"^{escaped}$")

    # --- Glob files: widen the tag position (POST -> *) to catch both PRIOR/POST ---
    glob_pattern = pattern.replace(right_tag, "*", 1)
    files = list(root.glob(glob_pattern))

    by_step: Dict[int, Dict[str, Path]] = {}
    for p in files:
        m = name_re.match(p.name)
        if not m:
            continue
        # Guard against malformed regex (should have 2 groups: timestep, tag)
        if m.lastindex is None or m.lastindex < 2:
            # Skip silently; regex didn't produce the expected groups
            continue

        step = int(m.group(1))
        tag = m.group(2)
        by_step.setdefault(step, {})[tag] = p

    pairs = [(s, d.get(left_tag), d.get(right_tag)) for s, d in by_step.items()]
    # Keep only complete pairs
    pairs = [(s, a, b) for (s, a, b) in pairs if a is not None and b is not None]
    pairs.sort(key=lambda x: x[0])

    saved_paths: List[Path] = []

    def build_out_name(post_name: str) -> str:
        """Replace the tag segment with concat_token using the compiled regex spans."""
        m = name_re.match(post_name)
        if not m or m.lastindex is None or m.lastindex < 2:
            # Fallback: a simple first-occurrence replacement
            return post_name.replace(right_tag, concat_token, 1)
        a, b = m.span(2)
        return post_name[:a] + concat_token + post_name[b:]

    def concat_horiz(im_left: Image.Image, im_right: Image.Image) -> Image.Image:
        """Concatenate two images horizontally (left then right)."""
        if im_left.height != im_right.height:
            scale = im_left.height / im_right.height
            new_w = max(1, int(round(im_right.width * scale)))
            im_right = im_right.resize((new_w, im_left.height), Image.BICUBIC)
        w = im_left.width + im_right.width
        h = max(im_left.height, im_right.height)
        canvas = Image.new("RGB", (w, h), (255, 255, 255))
        canvas.paste(im_left, (0, 0))
        canvas.paste(im_right, (im_left.width, 0))
        return canvas

    created = 0
    for step, prior_path, post_path in pairs:
        out_name = build_out_name(post_path.name)
        out_path = prior_path.parent / out_name

        if out_path.exists() and not overwrite:
            continue

        with Image.open(prior_path) as im_prior, Image.open(post_path) as im_post:
            im_prior = im_prior.convert("RGB")
            im_post = im_post.convert("RGB")
            im_cat = concat_horiz(im_prior, im_post)
            im_cat.save(out_path)

        created += 1
        saved_paths.append(out_path)

    return created, saved_paths

In [14]:
# --- CONFIGURE YOUR SETUP HERE ---
dataset = 'lorenz63'

image_directory = f"save/{dataset}_pf_vis" 
traj_index = 0
pattern = f"sigma_y1.0_batch64_len100_pfN1000000_timestep*_42_b{traj_index}_POST_0_fixed.png"

created, outputs = pair_prior_post(
    image_directory=image_directory,
    pattern=pattern,
)

In [ ]:

# --- CONFIGURE YOUR SETUP HERE ---
dataset = 'lorenz63'

image_directory = f"save/{dataset}_pf_vis" 
traj_index = 0
pattern = f"sigma_y1.0_batch64_len100_pfN1000000_timestep*_42_b1_POST_0_fixed.png"

frame_duration_ms = 200
max_timestep_to_include = 500 

output_gif_file = f"save/{dataset}_pf_vis/{dataset}_traj{traj_index}_{frame_duration_ms}ms_{max_timestep_to_include}timesteps.gif"

# --- END OF CONFIGURATION ---

# Run the function with your settings
create_gif_robust(
    image_folder=image_directory,
    file_pattern=pattern,
    gif_path=output_gif_file,
    duration=frame_duration_ms,
    max_value=max_timestep_to_include 
    )

Error: The file_pattern must contain exactly one '*' wildcard.


In [4]:

dataset = 'lorenz96'
image_directory = f"save/{dataset}_pf_vis" 
pattern = f"sigma_y1.0_batch64_len500_pfN1000000_timestep*_42_{traj_index}_adaptive.png"

frame_duration_ms = 400
max_value = 64

output_gif_file = f"save/{dataset}_pf_vis/{dataset}_zoomin_{frame_duration_ms}ms_{max_timestep_to_include}timesteps.gif"

# Run the function with your settings
create_gif_robust(
    image_folder=image_directory,
    file_pattern=pattern,
    gif_path=output_gif_file,
    duration=frame_duration_ms,
    max_value=max_value 
)

Filtering frames to include values up to 64.
Creating GIF with 64 frames...
GIF saved successfully at: save/lorenz96_pf_vis/lorenz96_zoomin_400ms_500timesteps.gif
